# Planner Node — Unit Tests

Exercises `planner_node` end-to-end with real inputs from `data/inputs/`.
One LLM call per run — costs a few cents at most with `gpt-4.1-mini`.

**Pre-requisites:**
1. `data/inputs/3gpp/28532-i00.md` (used only to feed the Loader before the Planner)
2. `data/inputs/rules/rules_bank_*.json`
3. `data/inputs/legacy/TS28532_ProvMnS.yaml` (optional — Scenario A uses it)
4. `OPENAI_API_KEY` in `.env`

The Planner does NOT use RAG, so Qdrant is irrelevant here.

In [ ]:
# Step 1 — Imports and path setup
#
# Test fixtures (which spec, rules, legacy to use) live in
# openapi_generator.config.paths, so the notebook stays free of glob magic
# and ad-hoc strings. Edit paths.py to switch fixtures.

import json
from pathlib import Path

import yaml

from openapi_generator.config import get_logger
from openapi_generator.config.paths import (
    TEST_LEGACY_PATH,
    TEST_RULES_PATH,
    TEST_SPEC_PATH,
)
from openapi_generator.nodes.loader import loader_node
from openapi_generator.nodes.planner import planner_node

logger = get_logger(__name__)

SPEC_PATH   = Path(TEST_SPEC_PATH).resolve()
RULES_PATH  = Path(TEST_RULES_PATH).resolve()
LEGACY_PATH = Path(TEST_LEGACY_PATH).resolve()

logger.info(f"Spec   : {SPEC_PATH}")
logger.info(f"Rules  : {RULES_PATH}")
logger.info(f"Legacy : {LEGACY_PATH if LEGACY_PATH.is_file() else '(not present)'}")
assert SPEC_PATH.is_file()
assert RULES_PATH.is_file()

In [ ]:
# Step 2 — Pre-parse rules_bank and legacy. Same trick as the loader notebook.

with open(RULES_PATH, "r", encoding="utf-8") as f:
    rules_bank = json.load(f)

legacy_openapi = None
if LEGACY_PATH.is_file():
    with open(LEGACY_PATH, "r", encoding="utf-8") as f:
        legacy_openapi = yaml.safe_load(f)

rules = rules_bank.get("rules", [])
logger.info(f"rules    : {len(rules)}")
logger.info(f"legacy   : {'present' if legacy_openapi else 'absent'}")
if legacy_openapi:
    logger.info(f"  paths   : {len(legacy_openapi.get('paths') or {})}")

In [3]:
# Step 3 — Run Loader first so the state is realistic (Planner is fed the
# Loader output, not raw inputs).

loader_state = {
    "spec_doc_path": str(SPEC_PATH),
    "rules_bank": rules_bank,
    "legacy_openapi": legacy_openapi,
}
loader_out = loader_node(loader_state)
logger.info(f"Loader produced {len(loader_out['parsed_spec_sections'])} sections")

# Compose the state for the Planner the way LangGraph would: Loader writes
# merged onto the initial state.
state_A = {**loader_state, **loader_out}

2026-05-25 22:37:03 [INFO] openapi_generator.nodes.loader: Loader → parsed 696 sections from 28532-i00.md (excluded 1 symbolic-title section(s))
2026-05-25 22:37:03 [INFO] openapi_generator.nodes.loader: Loader → rules_bank: 240 rule(s); legacy_openapi: present
2026-05-25 22:37:03 [INFO] openapi_generator.nodes.loader: Loader → seeding final_openapi from legacy (1 path(s), 16 schema(s))
2026-05-25 22:37:03 [INFO] __main__: Loader produced 696 sections


## Scenario A — Planner with legacy present

One LLM call. Expect a plan with a mix of `create` / `update` / `keep` since the legacy has 1 path and the rules describe many.

In [4]:
# Step 4 — Invoke Planner. Default LLM (ChatOpenAI from settings/.env).

out_A = planner_node(state_A)
ops = out_A["operations_plan"]

assert isinstance(ops, list)
assert len(ops) > 0, "Planner returned an empty operations_plan"
assert out_A["current_op_idx"] == 0
logger.info(f"Planner returned {len(ops)} operation(s).")

2026-05-25 22:37:04 [INFO] openapi_generator.nodes.planner: Planner Node started.
2026-05-25 22:37:04 [INFO] openapi_generator.config.llm_config: Default LLM ready: model=gpt-4.1-mini temperature=0.0
2026-05-25 22:38:10 [INFO] openapi_generator.nodes.planner: Planner Node complete — 25 operation(s): 21 create / 3 update / 1 keep; rules covered: 211/240
2026-05-25 22:38:10 [INFO] __main__: Planner returned 25 operation(s).


In [5]:
# Step 5 — Shape: each entry conforms to TargetOperation

REQUIRED_KEYS = {"path", "method", "action", "source_rule_ids", "priority", "rationale"}
ALLOWED_METHODS = {"get", "put", "post", "delete", "patch", "head", "options"}
ALLOWED_ACTIONS = {"create", "update", "keep"}
ALLOWED_PRIORITY = {"high", "medium", "low"}

for i, op in enumerate(ops):
    assert REQUIRED_KEYS.issubset(op.keys()), f"op {i} missing keys: {REQUIRED_KEYS - set(op.keys())}"
    assert op["method"] in ALLOWED_METHODS, f"op {i} bad method: {op['method']!r}"
    assert op["action"] in ALLOWED_ACTIONS, f"op {i} bad action: {op['action']!r}"
    assert op["priority"] in ALLOWED_PRIORITY, f"op {i} bad priority: {op['priority']!r}"
    assert isinstance(op["source_rule_ids"], list)
    for rid in op["source_rule_ids"]:
        assert isinstance(rid, int), f"op {i} non-int rule id: {rid!r}"
        assert 0 <= rid < len(rules), f"op {i} rule id {rid} out of range"

logger.info("Every operation conforms to the TargetOperation contract.")

2026-05-25 22:38:10 [INFO] __main__: Every operation conforms to the TargetOperation contract.


In [6]:
# Step 6 — No duplicate (path, method) pairs

pairs = [(op["path"], op["method"]) for op in ops]
assert len(pairs) == len(set(pairs)), f"duplicates found: {[p for p in pairs if pairs.count(p) > 1]}"
logger.info("No duplicate (path, method) pairs. OK.")

2026-05-25 22:38:10 [INFO] __main__: No duplicate (path, method) pairs. OK.


In [7]:
# Step 7 — Rule coverage report
# The prompt asks the LLM to cover every rule. In practice 100% coverage is
# aspirational; we log what was covered and what was missed so it's visible.

covered = {rid for op in ops for rid in op["source_rule_ids"]}
missing = sorted(set(range(len(rules))) - covered)

logger.info(f"Rule coverage: {len(covered)}/{len(rules)} ({100 * len(covered) // len(rules)}%)")
if missing:
    logger.warning(f"  {len(missing)} rule(s) not referenced. First few: {missing[:10]}")
    for idx in missing[:3]:
        r = rules[idx]
        logger.warning(
            f"    rule {idx}: section {r.get('section_id')} | "
            f"{r.get('rule_type')} | {(r.get('openapi_mapping') or {}).get('openapi_object')}"
        )

2026-05-25 22:38:10 [INFO] __main__: Rule coverage: 211/240 (87%)
2026-05-25 22:38:10 [WARNING] __main__:   29 rule(s) not referenced. First few: [85, 112, 113, 115, 117, 118, 119, 132, 133, 134]
2026-05-25 22:38:10 [WARNING] __main__:     rule 85: section 409 | path_parameter | paths./subscriptions/{subscriptionId}
2026-05-25 22:38:10 [WARNING] __main__:     rule 112: section 413 | schema_property | components/schemas/MergePatchAcknowledgeAlarms
2026-05-25 22:38:10 [WARNING] __main__:     rule 113: section 413 | schema_property | components/schemas/MergePatchAcknowledgeAlarms


In [8]:
# Step 8 — Action distribution and reuse of the legacy

from collections import Counter

by_action = Counter(op["action"] for op in ops)
by_priority = Counter(op["priority"] for op in ops)
logger.info(f"Action distribution  : {dict(by_action)}")
logger.info(f"Priority distribution: {dict(by_priority)}")

# Legacy paths should mostly map to update/keep, never to fresh 'create'
# (creating something that already exists would overwrite the legacy).
legacy_paths = set((legacy_openapi or {}).get("paths") or {})
creations_on_legacy = [op for op in ops if op["path"] in legacy_paths and op["action"] == "create"]
if creations_on_legacy:
    logger.warning(
        f"{len(creations_on_legacy)} operation(s) marked 'create' on a path that already "
        "exists in the legacy. Patcher may overwrite the legacy fragment."
    )
    for op in creations_on_legacy[:3]:
        logger.warning(f"  - {op['method']} {op['path']}")

2026-05-25 22:38:10 [INFO] __main__: Action distribution  : {'update': 3, 'keep': 1, 'create': 21}
2026-05-25 22:38:10 [INFO] __main__: Priority distribution: {'high': 2, 'medium': 17, 'low': 6}


In [9]:
# Step 9 — Inspect the first few operations

for op in ops[:5]:
    logger.info(
        f"  [{op['priority']:>6}] {op['action']:>6} {op['method'].upper():>6} "
        f"{op['path']}  (rules={op['source_rule_ids']})"
    )
    if op["rationale"]:
        logger.info(f"          {op['rationale']}")

2026-05-25 22:38:10 [INFO] __main__:   [  high] update    PUT /{className}={id}  (rules=[0, 1, 2, 3, 4, 5, 9, 10, 11, 12, 13, 17, 18, 21, 22, 23, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 209, 210, 211, 212, 213, 214, 215])
2026-05-25 22:38:10 [INFO] __main__:           This single PUT operation on /{className}={id} serves multiple IS operations including createMOI and modifyMOIAttributes, consolidating rules for creation and modification of managed object instances.
2026-05-25 22:38:10 [INFO] __main__:   [  high] update    GET /{className}={id}  (rules=[6, 7, 8, 14, 15, 16, 24, 28, 29, 30, 31, 32, 33, 34, 229, 230, 231, 232, 233, 228])
2026-05-25 22:38:10 [INFO] __main__:           GET operation on /{className}={id} reads managed object instances and includes query parameters and responses confirming and extending the legacy GET operation.
2026-05-25 22:38:10 [INFO] __main__:   [medium] update  PATCH /{class

## Scenario B — no legacy

Without a legacy, every operation should be `create` and the plan should still cover the rules. Costs another LLM call.

In [10]:
# Step 10 — Run Loader + Planner with legacy=None

loader_out_B = loader_node({"spec_doc_path": str(SPEC_PATH), "rules_bank": rules_bank, "legacy_openapi": None})
state_B = {
    "spec_doc_path": str(SPEC_PATH),
    "rules_bank": rules_bank,
    "legacy_openapi": None,
    **loader_out_B,
}
out_B = planner_node(state_B)
ops_B = out_B["operations_plan"]

assert len(ops_B) > 0
actions_B = Counter(op["action"] for op in ops_B)
logger.info(f"No-legacy plan: {len(ops_B)} operation(s); actions={dict(actions_B)}")

# Without a legacy, 'update' and 'keep' make no sense — flag if the LLM emits any.
non_create = [op for op in ops_B if op["action"] != "create"]
if non_create:
    logger.warning(
        f"{len(non_create)} operation(s) NOT marked 'create' despite no legacy — prompt may need tightening."
    )

2026-05-25 22:38:10 [INFO] openapi_generator.nodes.loader: Loader → parsed 696 sections from 28532-i00.md (excluded 1 symbolic-title section(s))
2026-05-25 22:38:10 [INFO] openapi_generator.nodes.loader: Loader → rules_bank: 240 rule(s); legacy_openapi: absent
2026-05-25 22:38:10 [INFO] openapi_generator.nodes.loader: Loader → no legacy provided; starting from empty skeleton (info from rules_bank: True)
2026-05-25 22:38:10 [INFO] openapi_generator.nodes.planner: Planner Node started.
2026-05-25 22:38:43 [INFO] openapi_generator.nodes.planner: Planner Node complete — 26 operation(s): 26 create / 0 update / 0 keep; rules covered: 175/240
2026-05-25 22:38:43 [INFO] __main__: No-legacy plan: 26 operation(s); actions={'create': 26}


## Failure-mode tests (no LLM call)

Empty rules → empty plan, no LLM invocation.

In [11]:
# Step 11 — rules_bank absent → empty plan, no crash, no LLM call

out_empty = planner_node({"rules_bank": {}, "legacy_openapi": None})
assert out_empty["operations_plan"] == []
assert out_empty["current_op_idx"] == 0
logger.info("Empty rules_bank → empty plan, no crash. OK.")

out_missing = planner_node({})
assert out_missing["operations_plan"] == []
logger.info("Missing rules_bank field → empty plan. OK.")

2026-05-25 22:38:43 [INFO] openapi_generator.nodes.planner: Planner Node started.
2026-05-25 22:38:43 [WARNING] openapi_generator.nodes.planner: Planner → rules_bank is empty; returning empty operations_plan
2026-05-25 22:38:43 [INFO] __main__: Empty rules_bank → empty plan, no crash. OK.
2026-05-25 22:38:43 [INFO] openapi_generator.nodes.planner: Planner Node started.
2026-05-25 22:38:43 [WARNING] openapi_generator.nodes.planner: Planner → rules_bank is empty; returning empty operations_plan
2026-05-25 22:38:43 [INFO] __main__: Missing rules_bank field → empty plan. OK.


In [12]:
# Step 12 — DI uniformity: Planner accepts retriever kwarg and ignores it

sentinel_ret = object()
out_di = planner_node({"rules_bank": {}}, retriever=sentinel_ret)
assert out_di["operations_plan"] == []
logger.info("Planner accepts retriever kwarg without using it — DI contract OK.")

2026-05-25 22:38:44 [INFO] openapi_generator.nodes.planner: Planner Node started.
2026-05-25 22:38:44 [WARNING] openapi_generator.nodes.planner: Planner → rules_bank is empty; returning empty operations_plan
2026-05-25 22:38:44 [INFO] __main__: Planner accepts retriever kwarg without using it — DI contract OK.
